In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")

# Convert timestamp to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Metric aggregates
metric_row_count = len(metric_df)
metric_cmdb_ids = sorted(metric_df['cmdb_id'].dropna().unique())
metric_kpi_names = sorted(metric_df['kpi_name'].dropna().unique())[:50]  # limit to top 50
metric_min_ts = metric_df['dt'].min()
metric_max_ts = metric_df['dt'].max()

metric_summary = pd.DataFrame({
    'rows': [metric_row_count],
    'distinct_cmdb_ids': [metric_cmdb_ids],
    'distinct_kpi_names_top50': [metric_kpi_names],
    'min_timestamp_Asia/Shanghai': [metric_min_ts],
    'max_timestamp_Asia/Shanghai': [metric_max_ts]
})

# Trace aggregates
trace_row_count = len(trace_df)
trace_cmdb_ids = sorted(trace_df['cmdb_id'].dropna().unique())
trace_names = sorted(trace_df['trace_name'].dropna().unique())[:50]  # limit to top 50
trace_min_ts = trace_df['dt'].min()
trace_max_ts = trace_df['dt'].max()

trace_summary = pd.DataFrame({
    'rows': [trace_row_count],
    'distinct_cmdb_ids': [trace_cmdb_ids],
    'distinct_trace_names_top50': [trace_names],
    'min_timestamp_Asia/Shanghai': [trace_min_ts],
    'max_timestamp_Asia/Shanghai': [trace_max_ts]
})

# Display compact summaries (DataFrames with one row each)
metric_summary, trace_summary

```
Out[1]:
```
```python
# Summarize based on previously computed metric_summary and trace_summary
summary = (
    f"metric.csv: {int(metric_summary.loc[0,'rows'])} rows; "
    f"distinct cmdb_id values: {metric_summary.loc[0,'distinct_cmdb_ids']}; "
    f"distinct kpi_name values (top {len(metric_summary.loc[0,'distinct_kpi_names_top50'])}): {metric_summary.loc[0,'distinct_kpi_names_top50']}; "
    f"time range (Asia/Shanghai): {metric_summary.loc[0,'min_timestamp_Asia/Shanghai']} to {metric_summary.loc[0,'max_timestamp_Asia/Shanghai']}. "
    f"trace.csv: {int(trace_summary.loc[0,'rows'])} rows; "
    f"distinct cmdb_id values: {trace_summary.loc[0,'distinct_cmdb_ids']}; "
    f"distinct trace_name values (top {len(trace_summary.loc[0,'distinct_trace_names_top50'])}): {trace_summary.loc[0,'distinct_trace_names_top50']}; "
    f"time range (Asia/Shanghai): {trace_summary.loc[0,'min_timestamp_Asia/Shanghai']} to {trace_summary.loc[0,'max_timestamp_Asia/Shanghai']}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    rows                                  distinct_cmdb_ids                           distinct_kpi_names_top50 min_timestamp_Asia/Shanghai max_timestamp_Asia/Shanghai
0  73357  [db_001, db_002, db_003, db_004, db_005, db_00...  [ACS, AIOS, AWS, Agent_ping, Asm_Free_Tb, Buff...   2020-05-29 03:00:00+08:00   2020-05-29 04:00:00+08:00,    rows                                  distinct_cmdb_ids                         distinct_trace_names_top50 min_timestamp_Asia/Shanghai max_timestamp_Asia/Shanghai
0  5536  [docker_001, docker_002, docker_003, docker_00...  [trace.from_docker_001.remoteprocess.duration_...   2020-05-29 03:30:00+08:00   2020-05-29 03:59:00+08:00)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure dt column with Asia/Shanghai tz exists
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in metric_df.columns or not pd.api.types.is_datetime64tz_dtype(metric_df['dt']):
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using entire metric_df
p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], sort=True)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter to failure window (Asia/Shanghai)
start = pd.Timestamp('2020-05-29 03:30:00', tz=tz)
end = pd.Timestamp('2020-05-29 04:00:00', tz=tz)
window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

# 3) For each series present in the window compute totals and comparisons against global P95
# Merge global p95 into window rows
merged = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Flag rows where value >= global p95
merged['ge_p95'] = merged['value'] >= merged['p95']

# total points per series in window
group_totals = (
    merged
    .groupby(['cmdb_id', 'kpi_name'], sort=True)
    .size()
    .rename('total_points_in_window')
    .reset_index()
)

# num points >= p95 per series
group_ge_counts = (
    merged[merged['ge_p95']]
    .groupby(['cmdb_id', 'kpi_name'], sort=True)
    .size()
    .rename('num_points_ge_P95')
    .reset_index()
)

# earliest timestamp in window where value >= p95
earliest_ts = (
    merged[merged['ge_p95']]
    .groupby(['cmdb_id', 'kpi_name'], sort=True)['dt']
    .min()
    .reset_index()
    .rename(columns={'dt': 'earliest_timestamp_in_window_where_value_ge_P95'})
)

# Assemble results: start from group_totals (only series present in window)
result = (
    group_totals
    .merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(group_ge_counts, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest_ts, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill missing counts with 0 and ensure earliest timestamps remain NaT where appropriate
result['num_points_ge_P95'] = result['num_points_ge_P95'].fillna(0).astype(int)
# Keep p95 as float; earliest timestamps are timezone-aware or NaT

# 4) Return compact table limited to top 20 rows ordered by num_points_ge_P95 desc
result_sorted = result.sort_values(['num_points_ge_P95', 'total_points_in_window'], ascending=[False, False])
result_top20 = result_sorted[
    ['cmdb_id', 'kpi_name', 'p95', 'total_points_in_window', 'num_points_ge_P95', 'earliest_timestamp_in_window_where_value_ge_P95']
].head(20)

# Total number of distinct (cmdb_id, kpi_name) series processed (i.e., present in the window)
total_series_processed = int(group_totals.shape[0])

# Display compact outputs
result_top20, total_series_processed

```
Out[2]:
```
```python
# Summarize results from previous computation into a plain-English string.
try:
    result_top20
    total_series_processed
except NameError:
    raise RuntimeError("Required variables result_top20 and total_series_processed not found in kernel state.")

# Prepare a short list of top entries (up to 5) for the summary
top_n = 5
rows = []
for _, r in result_top20.head(top_n).iterrows():
    ts = r['earliest_timestamp_in_window_where_value_ge_P95']
    ts_str = ts.isoformat() if pd.notna(ts) else 'null'
    rows.append(f"{r['cmdb_id']} | {r['kpi_name']} | p95={float(r['p95']):g} | {int(r['num_points_ge_P95'])}/{int(r['total_points_in_window'])} points >=P95 | earliest={ts_str}")

summary = (
    f"Processed {int(total_series_processed)} distinct (cmdb_id, kpi_name) series present in the window "
    f"(2020-05-29 03:30:00+08:00 through 2020-05-29 04:00:00+08:00).\n"
    f"Top {min(top_n, len(result_top20))} series by number of points >= global P95:\n"
    + "\n".join(rows)
    + "\n\nNote: Many top series reached >=P95 for all their points in the window; earliest exceedances are typically at 2020-05-29 03:30:00+08:00."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id         kpi_name         p95  total_points_in_window  num_points_ge_P95 earliest_timestamp_in_window_where_value_ge_P95
697   os_007      Disk_rd_ios    0.000000                      31                 31                       2020-05-29 03:30:00+08:00
1199  os_020     Disk_io_util    0.000000                      31                 31                       2020-05-29 03:30:00+08:00
10    db_001             Hang    1.000000                      30                 30                       2020-05-29 03:30:00+08:00
15    db_001        MEM_Total  381.902264                      30                 30                       2020-05-29 03:30:00+08:00
21    db_001     On_Off_State    1.000000                      30                 30                       2020-05-29 03:30:00+08:00
28    db_001         Row_Lock    0.000000                      30                 30                       2020-05-29 03:30:00+08:00
29    db_001  SctRead_Per_Sec    0.000000                      30                 30                       2020-05-29 03:30:00+08:00
44    db_002             AIOS    0.000000                      30                 30                       2020-05-29 03:30:00+08:00
53    db_002             Hang    1.000000                      30                 30                       2020-05-29 03:30:00+08:00
58    db_002        MEM_Total  381.902264                      30                 30                       2020-05-29 03:30:00+08:00
64    db_002     On_Off_State    1.000000                      30                 30                       2020-05-29 03:30:00+08:00
71    db_002         Row_Lock    0.000000                      30                 30                       2020-05-29 03:30:00+08:00
72    db_002  SctRead_Per_Sec    0.000000                      30                 30                       2020-05-29 03:30:00+08:00
96    db_003             Hang    1.000000                      30                 30                       2020-05-29 03:30:00+08:00
101   db_003        MEM_Total  381.902264                      30                 30                       2020-05-29 03:30:00+08:00
107   db_003     On_Off_State    1.000000                      30                 30                       2020-05-29 03:30:00+08:00
114   db_003         Row_Lock    0.000000                      30                 30                       2020-05-29 03:30:00+08:00
142   db_004             Hang    1.000000                      30                 30                       2020-05-29 03:30:00+08:00
146   db_004    Login_Per_Sec    1.200000                      30                 30                       2020-05-29 03:30:00+08:00
147   db_004        MEM_Total  381.902264                      30                 30                       2020-05-29 03:30:00+08:00, 1608)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure dt column with Asia/Shanghai tz exists
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in metric_df.columns or not pd.api.types.is_datetime64tz_dtype(metric_df['dt']):
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using entire metric_df
p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], sort=True)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter to failure window (Asia/Shanghai)
start = pd.Timestamp('2020-05-29 03:30:00', tz=tz)
end = pd.Timestamp('2020-05-29 04:00:00', tz=tz)
window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

# Merge global p95 into window rows
merged = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Flag rows where value >= global p95
merged['ge_p95'] = merged['value'] >= merged['p95']

# We'll detect consecutive-minute segments per (cmdb_id, kpi_name) where ge_p95 is True
segments = []  # collect segment dicts

# Process only groups that have at least one ge_p95 True row
grp = merged[merged['ge_p95']].sort_values(['cmdb_id', 'kpi_name', 'dt']).groupby(['cmdb_id', 'kpi_name'], sort=True)

for (cmdb_id, kpi_name), g in grp:
    # g is sorted by dt and contains only rows where ge_p95 == True
    if g.empty:
        continue
    # compute minute diffs between consecutive rows
    dt_diff_min = g['dt'].diff().dt.total_seconds().div(60).fillna(9999)  # large for first row
    # new segment starts where diff != 1
    segment_id = (dt_diff_min != 1).cumsum()
    g = g.assign(segment_id=segment_id.values)
    # aggregate per segment
    seg_agg = (
        g
        .groupby('segment_id', sort=True)
        .agg(
            segment_start=('dt', 'min'),
            segment_end=('dt', 'max'),
            segment_length_minutes=('dt', 'count'),
            segment_max_value=('value', 'max'),
            segment_mean_value=('value', 'mean')
        )
        .reset_index(drop=True)
    )
    # filter segments length >= 2 minutes
    seg_agg = seg_agg[seg_agg['segment_length_minutes'] >= 2]
    if seg_agg.empty:
        continue
    # attach cmdb_id, kpi_name, p95
    p95_val = float(p95_df[(p95_df['cmdb_id'] == cmdb_id) & (p95_df['kpi_name'] == kpi_name)]['p95'].iloc[0])
    seg_agg = seg_agg.assign(cmdb_id=cmdb_id, kpi_name=kpi_name, p95=p95_val)
    # reorder columns per requirement
    seg_agg = seg_agg[[
        'cmdb_id', 'kpi_name', 'p95',
        'segment_start', 'segment_end', 'segment_length_minutes',
        'segment_max_value', 'segment_mean_value'
    ]]
    # collect
    segments.append(seg_agg)

# Combine all segments
if segments:
    segments_df = pd.concat(segments, ignore_index=True)
else:
    segments_df = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'p95',
        'segment_start', 'segment_end', 'segment_length_minutes',
        'segment_max_value', 'segment_mean_value'
    ])

# 5) Prepare outputs
# (A) top 20 segments ordered by segment_length_minutes desc, then by segment_max_value desc
segments_sorted = segments_df.sort_values(['segment_length_minutes', 'segment_max_value'], ascending=[False, False])
segments_top20 = segments_sorted.head(20).reset_index(drop=True)

# Ensure timestamps are timezone-aware Asia/Shanghai (they should be)
# (B) scalar counts
total_segments_detected = int(segments_df.shape[0])
total_distinct_series_with_segments = int(segments_df[['cmdb_id', 'kpi_name']].drop_duplicates().shape[0])
total_distinct_cmdb_ids_with_segments = int(segments_df['cmdb_id'].drop_duplicates().shape[0])

# Compact outputs (DataFrame head and three scalars)
segments_top20, total_segments_detected, total_distinct_series_with_segments, total_distinct_cmdb_ids_with_segments

```
Out[3]:
```
```python
# Summarize the fault identification results in plain English.
try:
    segments_top20
    total_segments_detected
    total_distinct_series_with_segments
    total_distinct_cmdb_ids_with_segments
except NameError:
    raise RuntimeError("Required variables (segments_top20, total_segments_detected, total_distinct_series_with_segments, total_distinct_cmdb_ids_with_segments) not found in kernel state.")

# Build a concise summary: totals + top 5 segments
top_n = 5
lines = []
for _, r in segments_top20.head(top_n).iterrows():
    start = r['segment_start'].isoformat() if pd.notna(r['segment_start']) else 'null'
    end = r['segment_end'].isoformat() if pd.notna(r['segment_end']) else 'null'
    lines.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | length={int(r['segment_length_minutes'])}m | "
        f"max={float(r['segment_max_value']):g} | start={start} | end={end}"
    )

summary = (
    f"Fault detection summary for window 2020-05-29 03:30:00+08:00 to 2020-05-29 04:00:00+08:00:\n"
    f"- Total segments detected (length >=2 min): {int(total_segments_detected)}\n"
    f"- Total distinct (cmdb_id, kpi_name) series with segments: {int(total_distinct_series_with_segments)}\n"
    f"- Total distinct cmdb_id values with segments: {int(total_distinct_cmdb_ids_with_segments)}\n\n"
    f"Top {min(top_n, len(segments_top20))} segments (by length):\n" + "\n".join(lines) +
    "\n\nNote: Timestamps are in Asia/Shanghai (UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id          kpi_name           p95             segment_start               segment_end  segment_length_minutes  segment_max_value  segment_mean_value
0      os_007       Disk_rd_ios  0.000000e+00 2020-05-29 03:30:00+08:00 2020-05-29 04:00:00+08:00                      31       0.000000e+00        0.000000e+00
1      os_020      Disk_io_util  0.000000e+00 2020-05-29 03:30:00+08:00 2020-05-29 04:00:00+08:00                      31       0.000000e+00        0.000000e+00
2   redis_004         maxmemory  8.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       8.000000e+09        8.000000e+09
3   redis_008         maxmemory  8.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       8.000000e+09        8.000000e+09
4   redis_012         maxmemory  8.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       8.000000e+09        8.000000e+09
5   redis_004  used_memory_peak  3.027600e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       3.027600e+09        3.027600e+09
6   redis_012  used_memory_peak  3.006680e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       3.006680e+09        3.006680e+09
7   redis_008  used_memory_peak  2.945528e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.945528e+09        2.945528e+09
8   redis_004   used_memory_rss  2.531611e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.531611e+09        2.531611e+09
9   redis_008   used_memory_rss  2.529206e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.529206e+09        2.529206e+09
10  redis_001         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
11  redis_002         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
12  redis_003         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
13  redis_005         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
14  redis_006         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
15  redis_007         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
16  redis_009         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
17  redis_010         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
18  redis_011         maxmemory  2.000000e+09 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       2.000000e+09        2.000000e+09
19  redis_002  used_memory_peak  4.477894e+07 2020-05-29 03:30:00+08:00 2020-05-29 03:59:00+08:00                      30       4.477894e+07        4.477894e+07, 706, 513, 47)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure dt column with Asia/Shanghai tz exists
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in trace_df.columns or not pd.api.types.is_datetime64tz_dtype(trace_df['dt']):
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) using entire trace_df
p95_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], sort=True)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter to failure window (Asia/Shanghai)
start = pd.Timestamp('2020-05-29 03:30:00', tz=tz)
end = pd.Timestamp('2020-05-29 04:00:00', tz=tz)
trace_window = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end)].copy()

# 3) Merge global p95 into window and compute metrics
merged_trace = trace_window.merge(p95_trace, on=['cmdb_id', 'trace_name'], how='left')
merged_trace['ge_p95'] = merged_trace['value'] >= merged_trace['p95']

# total points per series in window
trace_totals = (
    merged_trace
    .groupby(['cmdb_id', 'trace_name'], sort=True)
    .size()
    .rename('total_points_in_window')
    .reset_index()
)

# num points >= p95 per series
trace_ge_counts = (
    merged_trace[merged_trace['ge_p95']]
    .groupby(['cmdb_id', 'trace_name'], sort=True)
    .size()
    .rename('num_points_ge_P95')
    .reset_index()
)

# earliest timestamp in window where value >= p95
trace_earliest = (
    merged_trace[merged_trace['ge_p95']]
    .groupby(['cmdb_id', 'trace_name'], sort=True)['dt']
    .min()
    .reset_index()
    .rename(columns={'dt': 'earliest_timestamp_in_window_where_value_ge_P95'})
)

# Assemble results
trace_result = (
    trace_totals
    .merge(p95_trace, on=['cmdb_id', 'trace_name'], how='left')
    .merge(trace_ge_counts, on=['cmdb_id', 'trace_name'], how='left')
    .merge(trace_earliest, on=['cmdb_id', 'trace_name'], how='left')
)

trace_result['num_points_ge_P95'] = trace_result['num_points_ge_P95'].fillna(0).astype(int)

# 4) Prepare compact table: top 20 by num_points_ge_P95 desc
trace_result_sorted = trace_result.sort_values(['num_points_ge_P95', 'total_points_in_window'], ascending=[False, False])
trace_top20 = trace_result_sorted[
    ['cmdb_id', 'trace_name', 'p95', 'total_points_in_window', 'num_points_ge_P95', 'earliest_timestamp_in_window_where_value_ge_P95']
].head(20)

# Scalar: total number of distinct (cmdb_id, trace_name) series processed (present in the window)
total_series_processed = int(trace_totals.shape[0])

# Display compact outputs
trace_top20, total_series_processed

```
Out[4]:
```
Summary (window 2020-05-29 03:30:00 to 2020-05-29 04:00:00, Asia/Shanghai UTC+8):

- Total distinct (cmdb_id, trace_name) series processed in the window: 200.
- Many trace series show saturated behavior vs their global P95: the top series all have 30/30 points in the window >= their global P95, with the earliest exceedance at 2020-05-29 03:30:00+08:00.
- Examples from the top results: docker_001 (trace.from_os_022.remoteprocess.error_rate, trace.self.csf.error_rate, trace.self.flyremote.error_rate, trace.self.jdbc.error_rate, trace.self.local.error_rate, etc.) — each had p95=0.0, total_points_in_window=30, num_points_ge_P95=30, earliest >=P95 = 2020-05-29 03:30:00+08:00.
- The top-20 table is dominated by docker_* series with full-window exceedances (30/30).  

Interpretation: a large set of trace error-rate metrics on docker nodes exceeded (or equaled) their global P95 for the entire 30-minute window, with issues starting at the window start (03:30+08:00).

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                    trace_name  p95  total_points_in_window  num_points_ge_P95 earliest_timestamp_in_window_where_value_ge_P95
2   docker_001    trace.from_os_022.remoteprocess.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
6   docker_001                     trace.self.csf.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
10  docker_001               trace.self.flyremote.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
14  docker_001                    trace.self.jdbc.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
18  docker_001                   trace.self.local.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
22  docker_001  trace.to_docker_007.remoteprocess.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
26  docker_001  trace.to_docker_008.remoteprocess.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
30  docker_002    trace.from_os_022.remoteprocess.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
42  docker_002                     trace.self.csf.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
46  docker_002               trace.self.flyremote.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
50  docker_002                    trace.self.jdbc.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
54  docker_002                   trace.self.local.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
58  docker_002  trace.to_docker_007.remoteprocess.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
62  docker_002  trace.to_docker_008.remoteprocess.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
66  docker_003    trace.from_os_021.remoteprocess.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
74  docker_003                     trace.self.csf.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
78  docker_003               trace.self.flyremote.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
82  docker_003                    trace.self.jdbc.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
86  docker_003                   trace.self.local.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00
90  docker_003  trace.to_docker_006.remoteprocess.error_rate  0.0                      30                 30                       2020-05-29 03:30:00+08:00, 200)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing segments_df if present; otherwise recompute segments following the same logic
try:
    segments_df
except NameError:
    # Ensure metric_df and p95_df exist or load/compute them
    try:
        metric_df
    except NameError:
        metric_df = pd.read_csv("metric.csv")
    # ensure dt column with Asia/Shanghai tz
    if 'dt' not in metric_df.columns or not pd.api.types.is_datetime64tz_dtype(metric_df['dt']):
        metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

    # compute global P95 per (cmdb_id, kpi_name)
    p95_df = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], sort=True)['value']
        .quantile(0.95)
        .reset_index(name='p95')
    )

    # filter to window
    start = pd.Timestamp('2020-05-29 03:30:00', tz=tz)
    end = pd.Timestamp('2020-05-29 04:00:00', tz=tz)
    window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

    # merge p95 and flag ge_p95
    merged = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
    merged['ge_p95'] = merged['value'] >= merged['p95']

    # detect consecutive-minute segments of ge_p95 per series with length >=2
    segments_list = []
    grp = merged[merged['ge_p95']].sort_values(['cmdb_id', 'kpi_name', 'dt']).groupby(['cmdb_id', 'kpi_name'], sort=True)
    for (cmdb_id, kpi_name), g in grp:
        if g.empty:
            continue
        dt_diff_min = g['dt'].diff().dt.total_seconds().div(60).fillna(9999)
        segment_id = (dt_diff_min != 1).cumsum()
        g = g.assign(segment_id=segment_id.values)
        seg_agg = (
            g
            .groupby('segment_id', sort=True)
            .agg(
                segment_start=('dt', 'min'),
                segment_end=('dt', 'max'),
                segment_length_minutes=('dt', 'count'),
                segment_max_value=('value', 'max'),
                segment_mean_value=('value', 'mean')
            )
            .reset_index(drop=True)
        )
        seg_agg = seg_agg[seg_agg['segment_length_minutes'] >= 2]
        if seg_agg.empty:
            continue
        p95_val = float(p95_df[(p95_df['cmdb_id'] == cmdb_id) & (p95_df['kpi_name'] == kpi_name)]['p95'].iloc[0])
        seg_agg = seg_agg.assign(cmdb_id=cmdb_id, kpi_name=kpi_name, p95=p95_val)
        seg_agg = seg_agg[[
            'cmdb_id', 'kpi_name', 'p95',
            'segment_start', 'segment_end', 'segment_length_minutes',
            'segment_max_value', 'segment_mean_value'
        ]]
        segments_list.append(seg_agg)

    if segments_list:
        segments_df = pd.concat(segments_list, ignore_index=True)
    else:
        segments_df = pd.DataFrame(columns=[
            'cmdb_id', 'kpi_name', 'p95',
            'segment_start', 'segment_end', 'segment_length_minutes',
            'segment_max_value', 'segment_mean_value'
        ])

# Ensure segment_start is timezone-aware Asia/Shanghai
if not segments_df.empty:
    if not pd.api.types.is_datetime64tz_dtype(segments_df['segment_start']):
        segments_df['segment_start'] = pd.to_datetime(segments_df['segment_start']).dt.tz_localize(tz)

# 2) Aggregate per cmdb_id
if segments_df.empty:
    agg_df = pd.DataFrame(columns=['cmdb_id', 'distinct_faulty_kpi_count', 'total_fault_minutes', 'earliest_segment_start'])
else:
    agg = (
        segments_df
        .groupby('cmdb_id', sort=True)
        .agg(
            distinct_faulty_kpi_count=('kpi_name', 'nunique'),
            total_fault_minutes=('segment_length_minutes', 'sum'),
            earliest_segment_start=('segment_start', 'min')
        )
        .reset_index()
    )
    agg_df = agg

# 3) Prepare final compact table (top 20 ordered by total_fault_minutes desc)
agg_sorted = agg_df.sort_values(['total_fault_minutes', 'distinct_faulty_kpi_count'], ascending=[False, False]).reset_index(drop=True)
agg_top20 = agg_sorted.head(20)

# Scalars
total_cmdb_ids_with_faults = int(agg_df.shape[0])
returned_cmdb_ids = list(agg_top20['cmdb_id'])

# Compact outputs
agg_top20, total_cmdb_ids_with_faults, returned_cmdb_ids

```
Out[5]:
```
```python
# Summarize aggregated fault segments per cmdb_id (compact plain-English)
try:
    agg_top20
    total_cmdb_ids_with_faults
    returned_cmdb_ids
except NameError:
    raise RuntimeError("Required variables not found in kernel state.")

top_n = 5
lines = []
for _, r in agg_top20.head(top_n).iterrows():
    start = r['earliest_segment_start']
    start_str = start.isoformat() if pd.notna(start) else 'null'
    lines.append(
        f"{r['cmdb_id']}: {int(r['distinct_faulty_kpi_count'])} faulty KPIs, "
        f"{int(r['total_fault_minutes'])} total fault-minutes, earliest start={start_str}"
    )

summary = (
    f"Fault aggregation (window 2020-05-29 03:30:00+08:00 to 2020-05-29 04:00:00+08:00):\n"
    f"- Total cmdb_ids with faults: {int(total_cmdb_ids_with_faults)}\n"
    f"- Top {min(top_n, len(agg_top20))} cmdb_ids by total_fault_minutes:\n"
    + "\n".join(lines)
    + "\n\n"
    f"- Returned table contains {len(returned_cmdb_ids)} cmdb_ids: {returned_cmdb_ids}\n"
    f"Note: timestamps shown are in Asia/Shanghai (UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  distinct_faulty_kpi_count  total_fault_minutes    earliest_segment_start
0   redis_004                         17                  430 2020-05-29 03:30:00+08:00
1   redis_008                         15                  423 2020-05-29 03:30:00+08:00
2   redis_012                         16                  405 2020-05-29 03:30:00+08:00
3   redis_010                         17                  388 2020-05-29 03:30:00+08:00
4   redis_011                         16                  378 2020-05-29 03:30:00+08:00
5   redis_006                         14                  367 2020-05-29 03:30:00+08:00
6   redis_007                         14                  365 2020-05-29 03:30:00+08:00
7   redis_003                         13                  363 2020-05-29 03:30:00+08:00
8   redis_005                         13                  363 2020-05-29 03:30:00+08:00
9      db_005                         14                  327 2020-05-29 03:30:00+08:00
10     db_011                         18                  323 2020-05-29 03:30:00+08:00
11     os_008                         11                  300 2020-05-29 03:30:00+08:00
12  redis_009                         14                  283 2020-05-29 03:30:00+08:00
13  redis_001                         13                  282 2020-05-29 03:30:00+08:00
14  redis_002                         12                  279 2020-05-29 03:30:00+08:00
15     os_007                         12                  276 2020-05-29 03:30:00+08:00
16     db_013                         12                  264 2020-05-29 03:30:00+08:00
17     os_006                         11                  258 2020-05-29 03:30:00+08:00
18     db_007                         14                  254 2020-05-29 03:30:00+08:00
19     db_006                         14                  253 2020-05-29 03:30:00+08:00, 47, ['redis_004', 'redis_008', 'redis_012', 'redis_010', 'redis_011', 'redis_006', 'redis_007', 'redis_003', 'redis_005', 'db_005', 'db_011', 'os_008', 'redis_009', 'redis_001', 'redis_002', 'os_007', 'db_013', 'os_006', 'db_007', 'db_006'])```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing aggregated per-cmdb_id table (agg_df) if present; otherwise recompute from segments_df or metric data
try:
    agg_df
except NameError:
    # If segments_df exists, aggregate it; else recompute segments then aggregate
    try:
        segments_df
    except NameError:
        # Recompute segments from metric.csv (global P95 over full series, window detection)
        metric_df = pd.read_csv("metric.csv")
        if 'dt' not in metric_df.columns or not pd.api.types.is_datetime64tz_dtype(metric_df['dt']):
            metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], sort=True)['value'].quantile(0.95).reset_index(name='p95')
        start = pd.Timestamp('2020-05-29 03:30:00', tz=tz)
        end = pd.Timestamp('2020-05-29 04:00:00', tz=tz)
        window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()
        merged = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
        merged['ge_p95'] = merged['value'] >= merged['p95']

        segments_list = []
        grp = merged[merged['ge_p95']].sort_values(['cmdb_id', 'kpi_name', 'dt']).groupby(['cmdb_id', 'kpi_name'], sort=True)
        for (cmdb_id, kpi_name), g in grp:
            if g.empty:
                continue
            dt_diff_min = g['dt'].diff().dt.total_seconds().div(60).fillna(9999)
            segment_id = (dt_diff_min != 1).cumsum()
            g = g.assign(segment_id=segment_id.values)
            seg_agg = (
                g
                .groupby('segment_id', sort=True)
                .agg(
                    segment_start=('dt', 'min'),
                    segment_end=('dt', 'max'),
                    segment_length_minutes=('dt', 'count'),
                    segment_max_value=('value', 'max'),
                    segment_mean_value=('value', 'mean')
                )
                .reset_index(drop=True)
            )
            seg_agg = seg_agg[seg_agg['segment_length_minutes'] >= 2]
            if seg_agg.empty:
                continue
            p95_val = float(p95_df[(p95_df['cmdb_id'] == cmdb_id) & (p95_df['kpi_name'] == kpi_name)]['p95'].iloc[0])
            seg_agg = seg_agg.assign(cmdb_id=cmdb_id, kpi_name=kpi_name, p95=p95_val)
            seg_agg = seg_agg[[
                'cmdb_id', 'kpi_name', 'p95',
                'segment_start', 'segment_end', 'segment_length_minutes',
                'segment_max_value', 'segment_mean_value'
            ]]
            segments_list.append(seg_agg)

        segments_df = pd.concat(segments_list, ignore_index=True) if segments_list else pd.DataFrame(columns=[
            'cmdb_id', 'kpi_name', 'p95',
            'segment_start', 'segment_end', 'segment_length_minutes',
            'segment_max_value', 'segment_mean_value'
        ])

    # Aggregate segments_df to produce agg_df
    if segments_df.empty:
        agg_df = pd.DataFrame(columns=['cmdb_id', 'distinct_faulty_kpi_count', 'total_fault_minutes', 'earliest_segment_start'])
    else:
        agg = (
            segments_df
            .groupby('cmdb_id', sort=True)
            .agg(
                distinct_faulty_kpi_count=('kpi_name', 'nunique'),
                total_fault_minutes=('segment_length_minutes', 'sum'),
                earliest_segment_start=('segment_start', 'min')
            )
            .reset_index()
        )
        agg_df = agg

# Build allowed cmdb_id lists
os_allowed = [f"os_{i:03d}" for i in range(1, 23)]
docker_allowed = [f"docker_{i:03d}" for i in range(1, 9)]
db_allowed = [f"db_{i:03d}" for i in range(1, 14)]
allowed_set = set(os_allowed + docker_allowed + db_allowed)

# Filter agg_df to allowed cmdb_ids only
agg_allowed = agg_df[agg_df['cmdb_id'].isin(allowed_set)].copy()

# If earliest_segment_start exists and is not tz-aware, ensure it's timezone-aware Asia/Shanghai
if not agg_allowed.empty:
    if not pd.api.types.is_datetime64tz_dtype(agg_allowed['earliest_segment_start']):
        agg_allowed['earliest_segment_start'] = pd.to_datetime(agg_allowed['earliest_segment_start']).dt.tz_localize(tz)

# Add level column (db/os/docker) by splitting cmdb_id
def get_level(cid):
    return cid.split('_')[0] if isinstance(cid, str) and '_' in cid else None

agg_allowed['level'] = agg_allowed['cmdb_id'].apply(get_level)

# 1) Per-level summary
level_summary = (
    agg_allowed
    .groupby('level', sort=False)
    .agg(
        total_distinct_cmdb_ids_with_faults=('cmdb_id', 'nunique'),
        sum_total_fault_minutes=('total_fault_minutes', 'sum'),
        sum_distinct_faulty_kpi_count=('distinct_faulty_kpi_count', 'sum')
    )
    .reset_index()
)

# Ensure integer types for counts
level_summary['total_distinct_cmdb_ids_with_faults'] = level_summary['total_distinct_cmdb_ids_with_faults'].astype(int)
level_summary['sum_total_fault_minutes'] = level_summary['sum_total_fault_minutes'].astype(int)
level_summary['sum_distinct_faulty_kpi_count'] = level_summary['sum_distinct_faulty_kpi_count'].astype(int)

# 2) Top allowed cmdb_ids (top 10 by total_fault_minutes desc)
top_allowed = (
    agg_allowed
    .assign(total_fault_minutes=lambda df: df['total_fault_minutes'].astype(int))
    .sort_values(['total_fault_minutes', 'distinct_faulty_kpi_count'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'level', 'distinct_faulty_kpi_count', 'total_fault_minutes', 'earliest_segment_start']]
    .head(10)
    .reset_index(drop=True)
)

# Compact outputs: level_summary and top_allowed
level_summary, top_allowed

```
Out[6]:
```
```python
# Build a plain-English summary from existing variables level_summary and top_allowed
try:
    level_summary
    top_allowed
except NameError:
    raise RuntimeError("Required variables level_summary and top_allowed not found in kernel state.")

# Format per-level summary
level_lines = []
for _, row in level_summary.iterrows():
    lvl = row['level']
    cnt = int(row['total_distinct_cmdb_ids_with_faults'])
    minutes = int(row['sum_total_fault_minutes'])
    kpis = int(row['sum_distinct_faulty_kpi_count'])
    level_lines.append(f"{lvl}: {cnt} cmdb_ids with faults, {minutes} total fault-minutes, {kpis} faulty KPI-count (sum)")

# Format top-10 cmdb_ids (allowed) table
top_lines = []
for _, r in top_allowed.iterrows():
    cid = r['cmdb_id']
    lvl = r['level']
    kcount = int(r['distinct_faulty_kpi_count'])
    minutes = int(r['total_fault_minutes'])
    start = r['earliest_segment_start']
    start_str = start.isoformat() if pd.notna(start) else 'null'
    top_lines.append(f"{cid} ({lvl}): {kcount} KPIs, {minutes} fault-minutes, earliest_start={start_str}")

summary = (
    "Aggregated fault summary (allowed cmdb_ids only, window 2020-05-29 03:30:00+08:00 to 2020-05-29 04:00:00+08:00):\n\n"
    "Per-level totals:\n- " + "\n- ".join(level_lines) + "\n\n"
    f"Top {len(top_allowed)} allowed cmdb_ids by total_fault_minutes:\n- " + "\n- ".join(top_lines)
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    level  total_distinct_cmdb_ids_with_faults  sum_total_fault_minutes  sum_distinct_faulty_kpi_count
0      db                                   13                     3101                            155
1  docker                                    7                     1370                             58
2      os                                   14                     2736                            125,   cmdb_id level  distinct_faulty_kpi_count  total_fault_minutes    earliest_segment_start
0  db_005    db                         14                  327 2020-05-29 03:30:00+08:00
1  db_011    db                         18                  323 2020-05-29 03:30:00+08:00
2  os_008    os                         11                  300 2020-05-29 03:30:00+08:00
3  os_007    os                         12                  276 2020-05-29 03:30:00+08:00
4  db_013    db                         12                  264 2020-05-29 03:30:00+08:00
5  os_006    os                         11                  258 2020-05-29 03:30:00+08:00
6  db_007    db                         14                  254 2020-05-29 03:30:00+08:00
7  db_006    db                         14                  253 2020-05-29 03:30:00+08:00
8  db_010    db                         11                  247 2020-05-29 03:30:00+08:00
9  db_012    db                         14                  244 2020-05-29 03:30:00+08:00)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df is loaded and has dt in Asia/Shanghai
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
if 'dt' not in metric_df.columns or not pd.api.types.is_datetime64tz_dtype(metric_df['dt']):
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Recompute global P95 per (cmdb_id, kpi_name) to be safe
p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], sort=True)['value'].quantile(0.95).reset_index(name='p95')

# Ensure segments_df exists; if not, recompute segments (window detection) including segment_max_value
try:
    segments_df
except NameError:
    # Filter to window
    start = pd.Timestamp('2020-05-29 03:30:00', tz=tz)
    end = pd.Timestamp('2020-05-29 04:00:00', tz=tz)
    window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()
    # merge p95 and flag ge_p95
    merged = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
    merged['ge_p95'] = merged['value'] >= merged['p95']
    # detect consecutive-minute segments of ge_p95 per series with length >=2
    segments_list = []
    grp = merged[merged['ge_p95']].sort_values(['cmdb_id', 'kpi_name', 'dt']).groupby(['cmdb_id', 'kpi_name'], sort=True)
    for (cmdb_id, kpi_name), g in grp:
        if g.empty:
            continue
        dt_diff_min = g['dt'].diff().dt.total_seconds().div(60).fillna(9999)
        segment_id = (dt_diff_min != 1).cumsum()
        g = g.assign(segment_id=segment_id.values)
        seg_agg = (
            g
            .groupby('segment_id', sort=True)
            .agg(
                segment_start=('dt', 'min'),
                segment_end=('dt', 'max'),
                segment_length_minutes=('dt', 'count'),
                segment_max_value=('value', 'max')
            )
            .reset_index(drop=True)
        )
        seg_agg = seg_agg[seg_agg['segment_length_minutes'] >= 2]
        if seg_agg.empty:
            continue
        seg_agg = seg_agg.assign(cmdb_id=cmdb_id, kpi_name=kpi_name)
        seg_agg = seg_agg[[
            'cmdb_id', 'kpi_name',
            'segment_start', 'segment_end', 'segment_length_minutes', 'segment_max_value'
        ]]
        segments_list.append(seg_agg)
    segments_df = pd.concat(segments_list, ignore_index=True) if segments_list else pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'segment_start', 'segment_end', 'segment_length_minutes', 'segment_max_value'
    ])

# Aggregate to get total_fault_minutes per cmdb_id and pick top 10 db_* candidates
if segments_df.empty:
    top_db_ids = []
    severity_top50 = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'p95', 'p95_zero',
        'segment_max_value', 'breach_ratio', 'segment_start', 'segment_length_minutes'
    ])
else:
    agg = (
        segments_df
        .groupby('cmdb_id', sort=True)
        .agg(total_fault_minutes=('segment_length_minutes', 'sum'))
        .reset_index()
    )
    db_agg = agg[agg['cmdb_id'].str.startswith('db_')].copy()
    db_agg['total_fault_minutes'] = db_agg['total_fault_minutes'].astype(int)
    top_db_candidates = db_agg.sort_values('total_fault_minutes', ascending=False).head(10)
    top_db_ids = list(top_db_candidates['cmdb_id'])

    # Filter segments for these top db ids
    segments_top_dbs = segments_df[segments_df['cmdb_id'].isin(top_db_ids)].copy()
    # Ensure segment_start is timezone-aware Asia/Shanghai
    if not segments_top_dbs.empty and not pd.api.types.is_datetime64tz_dtype(segments_top_dbs['segment_start']):
        segments_top_dbs['segment_start'] = pd.to_datetime(segments_top_dbs['segment_start']).dt.tz_localize(tz)

    # Merge with global p95 per (cmdb_id,kpi_name)
    seg_with_p95 = segments_top_dbs.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

    # Ensure p95 column exists
    if 'p95' not in seg_with_p95.columns:
        seg_with_p95['p95'] = np.nan

    # Compute p95_zero flag and breach_ratio
    seg_with_p95['p95_zero'] = seg_with_p95['p95'] == 0
    seg_with_p95['breach_ratio'] = np.where(
        seg_with_p95['p95_zero'],
        np.inf,
        seg_with_p95['segment_max_value'] / seg_with_p95['p95'].replace({0: np.nan})
    )

    # Prepare final table and sort by breach_ratio desc
    final_cols = [
        'cmdb_id', 'kpi_name', 'p95', 'p95_zero',
        'segment_max_value', 'breach_ratio', 'segment_start', 'segment_length_minutes'
    ]
    severity_table = seg_with_p95.loc[:, final_cols].copy()
    severity_table = severity_table.sort_values(['breach_ratio', 'segment_max_value'], ascending=[False, False]).reset_index(drop=True)
    severity_top50 = severity_table.head(50)

# Outputs: severity_top50 (compact table) and top_db_ids (list of analyzed db ids)
severity_top50, top_db_ids

```
Out[7]:
```
```python
# Summarize severity results for top DB candidates
try:
    severity_top50
    top_db_ids
except NameError:
    raise RuntimeError("Required variables severity_top50 and top_db_ids not found in kernel state.")

# Compact stats
total_rows = int(severity_top50.shape[0])
p95_missing = int(severity_top50['p95'].isna().sum()) if 'p95' in severity_top50.columns else 0
p95_zero_count = int(severity_top50['p95_zero'].sum()) if 'p95_zero' in severity_top50.columns else 0
finite_breach = int(np.isfinite(severity_top50['breach_ratio']).sum()) if 'breach_ratio' in severity_top50.columns else 0

# Build example top entries (up to 5) for summary
examples = []
for _, r in severity_top50.head(5).iterrows():
    br = r['breach_ratio']
    if pd.isna(br):
        br_str = 'N/A (p95 missing)'
    elif np.isinf(br):
        br_str = 'INF (p95==0)'
    else:
        br_str = f"{float(br):.2f}"
    start = r['segment_start'].isoformat() if pd.notna(r['segment_start']) else 'null'
    examples.append(f"{r['cmdb_id']} | {r['kpi_name']} | p95={r['p95']} | max={r['segment_max_value']:.2f} | breach={br_str} | start={start} | len={int(r['segment_length_minutes'])}m")

summary = (
    f"Severity analysis for top DBs (top-10 analyzed: {top_db_ids}):\n"
    f"- Rows produced: {total_rows} (top 50 returned)\n"
    f"- p95 missing for {p95_missing} rows; p95==0 flagged for {p95_zero_count} rows\n"
    f"- breach_ratio finite for {finite_breach} rows; otherwise unavailable\n"
    f"- Example top entries (by breach_ratio):\n  " + "\n  ".join(examples) + "\n\n"
    "Notes: breach_ratio = segment_max_value / global_P95. If global P95 is missing, breach_ratio is N/A. "
    "If p95==0, breach_ratio is treated as infinite and p95_zero is True. Timestamps are Asia/Shanghai (UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id             kpi_name  p95  p95_zero  segment_max_value  breach_ratio             segment_start  segment_length_minutes
0   db_009         Redo_Per_Sec  NaN     False           34521.70           NaN 2020-05-29 03:33:00+08:00                       2
1   db_013       Total_Tbs_Size  NaN     False            4094.00           NaN 2020-05-29 03:51:00+08:00                       2
2   db_012        Used_Tbs_Size  NaN     False            2529.55           NaN 2020-05-29 03:51:00+08:00                       2
3   db_011       Total_Tbs_Size  NaN     False            2172.00           NaN 2020-05-29 03:39:00+08:00                       2
4   db_006         Call_Per_Sec  NaN     False             550.40           NaN 2020-05-29 03:52:00+08:00                       2
..     ...                  ...  ...       ...                ...           ...                       ...                     ...
45  db_012          TPS_Per_Sec  NaN     False              15.50           NaN 2020-05-29 03:51:00+08:00                       2
46  db_009        Proc_Used_Pct  NaN     False              13.88           NaN 2020-05-29 03:42:00+08:00                       2
47  db_009   Proc_User_Used_Pct  NaN     False              13.88           NaN 2020-05-29 03:42:00+08:00                       2
48  db_002  tnsping_result_time  NaN     False              10.00           NaN 2020-05-29 03:40:00+08:00                       2
49  db_002  tnsping_result_time  NaN     False              10.00           NaN 2020-05-29 03:51:00+08:00                       4

[50 rows x 8 columns], ['db_005', 'db_011', 'db_013', 'db_007', 'db_006', 'db_010', 'db_012', 'db_002', 'db_009', 'db_004'])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure dt column with Asia/Shanghai tz exists
if 'dt' not in metric_df.columns or not pd.api.types.is_datetime64tz_dtype(metric_df['dt']):
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Settings
db_allowed = [f"db_{i:03d}" for i in range(1, 14)]
kpi_list = ['Hang', 'On_Off_State', 'MEM_Total', 'Login_Per_Sec']

# 1) Compute global P95 for each (cmdb_id, kpi_name) for allowed DBs and selected KPIs (global over full metric_df)
p95_db_kpis = (
    metric_df[
        metric_df['cmdb_id'].isin(db_allowed) & metric_df['kpi_name'].isin(kpi_list)
    ]
    .groupby(['cmdb_id', 'kpi_name'], sort=True)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter metric_df to the failure window (inclusive)
start = pd.Timestamp('2020-05-29 03:30:00', tz=tz)
end = pd.Timestamp('2020-05-29 04:00:00', tz=tz)
window_db = metric_df[
    (metric_df['dt'] >= start) &
    (metric_df['dt'] <= end) &
    (metric_df['cmdb_id'].isin(db_allowed)) &
    (metric_df['kpi_name'].isin(kpi_list))
].copy()

# 3) For each (cmdb_id, kpi_name) present in the window compute required fields
# Merge global p95 into window rows (p95 computed from full series as required)
window_with_p95 = window_db.merge(p95_db_kpis, on=['cmdb_id', 'kpi_name'], how='left')

# Define event boolean per rules:
# - For 'Hang' and 'On_Off_State' count points where value == 1
# - For 'MEM_Total' and 'Login_Per_Sec' count points where value >= p95 (if p95 is NaN => event False)
def compute_event_flag(df):
    cond_hang = (df['kpi_name'].isin(['Hang', 'On_Off_State'])) & (df['value'] == 1)
    # For MEM_Total and Login_Per_Sec, require p95 not null and value >= p95
    cond_ge_p95 = (
        df['kpi_name'].isin(['MEM_Total', 'Login_Per_Sec']) &
        df['p95'].notna() &
        (df['value'] >= df['p95'])
    )
    return cond_hang | cond_ge_p95

window_with_p95['event_flag'] = compute_event_flag(window_with_p95)

# Aggregations per (cmdb_id, kpi_name)
agg = (
    window_with_p95
    .groupby(['cmdb_id', 'kpi_name'], sort=True)
    .agg(
        p95=('p95', 'first'),  # global p95
        total_points_in_window=('value', 'size'),
        event_count=('event_flag', 'sum'),
        earliest_event_timestamp_in_window=('dt', lambda s: s[window_with_p95.loc[s.index, 'event_flag']].min() if s.size>0 else pd.NaT)
    )
    .reset_index()
)

# If earliest_event_timestamp_in_window is NaT, keep as NaT (will display null)
# Ensure earliest_event_timestamp_in_window is tz-aware (should be already)
# 4) Return compact table limited to 20 rows ordered by event_count desc then by cmdb_id
agg_sorted = agg.sort_values(['event_count', 'cmdb_id'], ascending=[False, True]).reset_index(drop=True)
agg_top20 = agg_sorted[[
    'cmdb_id', 'kpi_name', 'p95', 'total_points_in_window', 'event_count', 'earliest_event_timestamp_in_window'
]].head(20)

# Scalar: number of db cmdb_ids that had any event_count > 0
dbs_with_events = agg[agg['event_count'] > 0]['cmdb_id'].nunique()

# Compact outputs
agg_top20, int(dbs_with_events)

```
Out[8]:
```
```python
# Summarize the results for selected DB KPIs in plain English using previously computed agg_top20 and dbs_with_events
try:
    agg_top20
    dbs_with_events
except NameError:
    raise RuntimeError("Required variables agg_top20 and dbs_with_events not found in kernel state.")

# Build short examples from the returned table (up to 8 rows)
examples = []
for _, r in agg_top20.head(8).iterrows():
    cid = r['cmdb_id']
    k = r['kpi_name']
    p95 = r['p95']
    total = int(r['total_points_in_window'])
    ev = int(r['event_count'])
    ts = r['earliest_event_timestamp_in_window']
    ts_str = ts.isoformat() if pd.notna(ts) else 'null'
    examples.append(f"{cid} | {k} | p95={p95} | points={total} | events={ev} | earliest={ts_str}")

summary = (
    "DB KPI event summary (KPIs: Hang, On_Off_State, MEM_Total, Login_Per_Sec) for window "
    "2020-05-29 03:30:00+08:00 through 2020-05-29 04:00:00+08:00:\n\n"
    f"- DB components checked: db_001 .. db_013\n"
    f"- Number of DB cmdb_ids with any events (event_count>0): {int(dbs_with_events)}\n\n"
    "Key observations from the returned rows:\n"
    "- For most DBs, 'Hang', 'On_Off_State', and 'MEM_Total' each had event_count = 30 (i.e., all minutes in the 30-min window),\n"
    "  with earliest events at 2020-05-29 03:30:00+08:00.\n"
    "- 'Login_Per_Sec' appears as an event (>=P95) for db_004 in the window (also 30 points).\n\n"
    "Example entries (cmdb_id | kpi_name | p95 | points | events | earliest_event):\n  "
    + "\n  ".join(examples)
    + "\n\nNote: 'event_count' for 'Hang' and 'On_Off_State' counts value==1; for 'MEM_Total' and 'Login_Per_Sec' counts value>=global P95. "
    "Timestamps are Asia/Shanghai (UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id       kpi_name         p95  total_points_in_window  event_count earliest_event_timestamp_in_window
0   db_001           Hang    1.000000                      30           30          2020-05-29 03:30:00+08:00
1   db_001      MEM_Total  381.902264                      30           30          2020-05-29 03:30:00+08:00
2   db_001   On_Off_State    1.000000                      30           30          2020-05-29 03:30:00+08:00
3   db_002           Hang    1.000000                      30           30          2020-05-29 03:30:00+08:00
4   db_002      MEM_Total  381.902264                      30           30          2020-05-29 03:30:00+08:00
5   db_002   On_Off_State    1.000000                      30           30          2020-05-29 03:30:00+08:00
6   db_003           Hang    1.000000                      30           30          2020-05-29 03:30:00+08:00
7   db_003      MEM_Total  381.902264                      30           30          2020-05-29 03:30:00+08:00
8   db_003   On_Off_State    1.000000                      30           30          2020-05-29 03:30:00+08:00
9   db_004           Hang    1.000000                      30           30          2020-05-29 03:30:00+08:00
10  db_004  Login_Per_Sec    1.200000                      30           30          2020-05-29 03:30:00+08:00
11  db_004      MEM_Total  381.902264                      30           30          2020-05-29 03:30:00+08:00
12  db_004   On_Off_State    1.000000                      30           30          2020-05-29 03:30:00+08:00
13  db_005           Hang    1.000000                      30           30          2020-05-29 03:30:00+08:00
14  db_005      MEM_Total  506.929929                      30           30          2020-05-29 03:30:00+08:00
15  db_005   On_Off_State    1.000000                      30           30          2020-05-29 03:30:00+08:00
16  db_006           Hang    1.000000                      30           30          2020-05-29 03:30:00+08:00
17  db_006      MEM_Total  508.252410                      30           30          2020-05-29 03:30:00+08:00
18  db_006   On_Off_State    1.000000                      30           30          2020-05-29 03:30:00+08:00
19  db_007           Hang    1.000000                      30           30          2020-05-29 03:30:00+08:00, 13)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure dt column with Asia/Shanghai tz exists
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in trace_df.columns or not pd.api.types.is_datetime64tz_dtype(trace_df['dt']):
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Keywords (case-insensitive)
keywords = ['jdbc', 'db', 'tnsping', 'sql', 'duration', 'remoteprocess']
pattern = "(" + "|".join(re.escape(k) for k in keywords) + ")"

# 1) Identify matched (cmdb_id, trace_name) series
mask_keyword = trace_df['trace_name'].str.contains(pattern, case=False, regex=True, na=False)
matched_pairs = trace_df[mask_keyword][['cmdb_id', 'trace_name']].drop_duplicates().reset_index(drop=True)

# Scalar: total number of matched series
total_series_processed = int(matched_pairs.shape[0])

# 1) Compute global P95 per matched pair using full trace_df (global, BEFORE window filter)
p95_trace_matched = (
    trace_df[mask_keyword]
    .groupby(['cmdb_id', 'trace_name'], sort=True)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter trace_df to the failure window (Asia/Shanghai)
start = pd.Timestamp('2020-05-29 03:30:00', tz=tz)
end = pd.Timestamp('2020-05-29 04:00:00', tz=tz)
trace_window = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end)].copy()

# Keep only matched series in the window
trace_window_matched = trace_window[trace_window['trace_name'].str.contains(pattern, case=False, regex=True, na=False)].copy()

# 3) For each matched series present in the window compute aggregates and earliest >=p95
# Merge p95 into window rows
merged = trace_window_matched.merge(p95_trace_matched, on=['cmdb_id', 'trace_name'], how='left')

# Flag where value >= p95 (handle NaN p95 -> False)
merged['ge_p95'] = merged['p95'].notna() & (merged['value'] >= merged['p95'])

# Aggregations per series present in the window
agg_window = (
    merged
    .groupby(['cmdb_id', 'trace_name'], sort=True)
    .agg(
        p95=('p95', 'first'),
        total_points_in_window=('value', 'size'),
        num_points_ge_P95=('ge_p95', 'sum'),
        earliest_timestamp_in_window_where_value_ge_P95=('dt', lambda s: s[merged.loc[s.index, 'ge_p95']].min() if s.size>0 else pd.NaT)
    )
    .reset_index()
)

# p95_zero flag
agg_window['p95_zero'] = agg_window['p95'] == 0

# Ensure earliest timestamp is tz-aware or NaT
# 4) Ordering: exclude p95==0 from top ordering unless no non-zero-p95 series exist
non_zero = agg_window[~agg_window['p95_zero']].copy()
if non_zero.shape[0] > 0:
    candidates = non_zero
else:
    candidates = agg_window.copy()

# Order by num_points_ge_P95 desc, then total_points_in_window desc
candidates_sorted = candidates.sort_values(['num_points_ge_P95', 'total_points_in_window'], ascending=[False, False])

# Limit to top 20
matched_top20 = candidates_sorted[
    ['cmdb_id', 'trace_name', 'p95', 'p95_zero', 'total_points_in_window', 'num_points_ge_P95', 'earliest_timestamp_in_window_where_value_ge_P95']
].head(20).reset_index(drop=True)

# Final compact outputs: matched_top20 and total_series_processed
matched_top20, total_series_processed

```
Out[9]:
```
Summary (window 2020-05-29 03:30–04:00, Asia/Shanghai):

- Total matched trace series (trace_name containing jdbc, db, tnsping, sql, duration, or remoteprocess): 156.
- Ordering excluded series with p95 == 0 where possible; the top-20 shown contain non-zero p95 values (p95_zero = False).

Top findings (examples from the top results ordered by num_points_ge_P95):
- docker_006 | trace.self.jdbc.duration_p95 — p95=0.003, 30/30 points ≥ p95, earliest ≥p95: 2020-05-29 03:30:00+08:00
- docker_008 | trace.self.jdbc.duration_p95 — p95=0.003, 30/30 points ≥ p95, earliest: 2020-05-29 03:30:00+08:00
- docker_008 | trace.self.local.duration_p95 — p95=0.009, 18/30 points ≥ p95, earliest: 2020-05-29 03:33:00+08:00
- docker_002 | trace.self.jdbc.duration_p95 — p95=0.004, 4/30 points ≥ p95, earliest: 2020-05-29 03:32:00+08:00
- docker_001 and others — multiple JDBC/remoteprocess/duration traces with 2–3 points ≥ p95, earliest exceedances between ~03:32 and 03:45+08:00.

Interpretation:
- Several docker nodes (notably docker_006 and docker_008) show saturated JDBC-duration traces across the entire 30-minute window (30/30 points ≥ global P95), indicating sustained elevated trace durations related to DB access.
- Other docker nodes show partial breaches (fewer points ≥ P95), with earliest breaches often shortly after the window start.
- All timestamps reported are Asia/Shanghai (UTC+8).

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                        trace_name          p95  p95_zero  total_points_in_window  num_points_ge_P95 earliest_timestamp_in_window_where_value_ge_P95
0   docker_006                      trace.self.jdbc.duration_p95     0.003000     False                      30                 30                       2020-05-29 03:30:00+08:00
1   docker_008                      trace.self.jdbc.duration_p95     0.003000     False                      30                 30                       2020-05-29 03:30:00+08:00
2   docker_008                     trace.self.local.duration_p95     0.009000     False                      30                 18                       2020-05-29 03:33:00+08:00
3   docker_002                      trace.self.jdbc.duration_p95     0.004000     False                      30                  4                       2020-05-29 03:32:00+08:00
4   docker_001    trace.to_docker_008.remoteprocess.duration_p95     0.038000     False                      30                  3                       2020-05-29 03:44:00+08:00
5   docker_003                      trace.self.jdbc.duration_p95     0.005000     False                      30                  3                       2020-05-29 03:32:00+08:00
6   docker_008  trace.from_docker_001.remoteprocess.duration_p95     0.038000     False                      30                  3                       2020-05-29 03:44:00+08:00
7   docker_001     trace.from_os_022.remoteprocess.duration_mean    21.364414     False                      30                  2                       2020-05-29 03:45:00+08:00
8   docker_001      trace.from_os_022.remoteprocess.duration_p95    29.784570     False                      30                  2                       2020-05-29 03:42:00+08:00
9   docker_001         trace.from_os_022.remoteprocess.row_count   109.650000     False                      30                  2                       2020-05-29 03:40:00+08:00
10  docker_001                      trace.self.csf.duration_mean     0.593907     False                      30                  2                       2020-05-29 03:42:00+08:00
11  docker_001                       trace.self.csf.duration_p95     1.515097     False                      30                  2                       2020-05-29 03:42:00+08:00
12  docker_001                trace.self.flyremote.duration_mean     0.175758     False                      30                  2                       2020-05-29 03:45:00+08:00
13  docker_001                 trace.self.flyremote.duration_p95     0.329840     False                      30                  2                       2020-05-29 03:45:00+08:00
14  docker_001                     trace.self.jdbc.duration_mean     0.069687     False                      30                  2                       2020-05-29 03:45:00+08:00
15  docker_001                      trace.self.jdbc.duration_p95     0.198550     False                      30                  2                       2020-05-29 03:45:00+08:00
16  docker_001                         trace.self.jdbc.row_count  1754.400000     False                      30                  2                       2020-05-29 03:40:00+08:00
17  docker_001                    trace.self.local.duration_mean     0.534952     False                      30                  2                       2020-05-29 03:45:00+08:00
18  docker_001                     trace.self.local.duration_p95     1.455450     False                      30                  2                       2020-05-29 03:42:00+08:00
19  docker_001   trace.to_docker_007.remoteprocess.duration_mean     0.030089     False                      30                  2                       2020-05-29 03:43:00+08:00, 156)```
```